# 台股五檔 LSTM 原始模型補齊版

依《東吳大學資科系 專題計畫構想書 修正版》的**舊版對照組**補齊：
以過去 10 個交易日的收盤價建立單變量 LSTM，五檔股票各配置 20%，
完成歷史 150 日模型檢驗與固定持有回測。保留原始八個分析段落。

標的：台積電 2330、鴻海 2317、長榮 2603、中鋼 2002、中信金 2891；大盤基準為 ^TWII。

**執行方式：**在 Google Colab 上傳此檔案，依序執行全部儲存格。
第一次會安裝套件、下載股價、訓練六個歷史測試模型與六個未來推論模型。
後續段落共用快取，不會為每個指標重複訓練。

**目前驗證狀態：**已檢查 Python 語法、時間切分、Scaler、交易成本、MDD、
資料對齊與函式串接。建置環境沒有 TensorFlow / yfinance，未執行真實 LSTM 訓練與行情下載。
此檔案不含先前版本的舊輸出；表格數值會在你執行後產生。

**本次範圍：**先完成原始對照組。新聞情緒與動態權重由後續模組串接，
三方策略比較表的新版欄位保持空值並標示待串接，不填入構想書的預期成效。

## 0 套件安裝

Colab 已有多數套件。安裝缺少的套件後，若 Colab 提示必須重新啟動工作階段，
重啟後從「套件載入與參數設定」繼續。使用本機 Cursor 時，可在 Python 環境安裝同一組套件。

In [ ]:
%pip install -q tensorflow yfinance numpy pandas matplotlib scikit-learn joblib exchange-calendars

## 套件載入與參數設定

- 沿用原始雙層 50 單元 LSTM、Dropout 0.2、10 日視窗、10 萬元本金與五檔各 20%。
- `END_DATE_GLOBAL = "2026-10-14"` 是不包含截止日的設定，代表讀取至 2026-10-13 的可用歷史。
  若要重現原始 2025 年資料區間，改成 `"2025-10-31"`。
- `TEST_DAYS = 150`、`VALIDATION_DAYS = 60`；至少另外保留 80 個訓練樣本。
  若資料太短會明確停止，不會偷偷縮短測試期間或以缺失資料填零。
- `AUTO_ADJUST = False` 使用原始收盤價。此價格回測未納入股息再投資；新舊比較要採相同設定。
- 手續費 0.1425% 與賣出稅 0.3% 是可調整的研究參數。此版採持有後賣出，並未模擬當沖。
- `LIQUIDATE_AT_END = True` 表示在回測最後一日結算持股並扣賣出成本。
- 若要離線重跑，設定 `LOCAL_PRICE_CSV` 為之前輸出的 `historical_prices.csv`。
  檔案需同時包含五檔與 ^TWII；新聞特徵檔只有大盤波動率，不能替代大盤收盤價。
- 預測日期使用 `exchange-calendars` 的 XTAI 日曆。臨時休市可填入 `EXTRA_CLOSED_DATES`。

參數更改後請從此儲存格重新依序執行，避免使用先前的快取。
> GitHub 每日串接版：截止日期由 LSTM_AS_OF_DATE 注入；不得把原固定日期拿去當今天結果。


In [ ]:
"""構想書的舊版對照組：單變量 LSTM 與固定等權重投資組合。"""
import os
os.environ.setdefault("TF_DETERMINISTIC_OPS", "1")
import hashlib
import json
import random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import joblib

try:
    import yfinance as yf
except ImportError:
    yf = None
try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
    from tensorflow.keras.callbacks import EarlyStopping
except ImportError:
    tf = None
    EarlyStopping = None
try:
    import exchange_calendars as xcals
except ImportError:
    xcals = None
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value.to_string() if hasattr(value, "to_string") else value)

PORTFOLIO_CONFIG = {"2330.TW": .2, "2002.TW": .2, "2603.TW": .2,
                    "2891.TW": .2, "2317.TW": .2}
STOCK_NAMES = {"2330.TW": "台積電", "2002.TW": "中鋼", "2603.TW": "長榮",
               "2891.TW": "中信金", "2317.TW": "鴻海", "^TWII": "加權指數"}
MARKET_TICKER = "^TWII"
INITIAL_CAPITAL = 100000.0
START_DATE_GLOBAL = os.environ.get("LSTM_START_DATE", "2014-01-01")
# yfinance 的 end 不包含當天。本次固定截止日，方便日後重現相同結果。
END_DATE_GLOBAL = (pd.Timestamp(os.environ["LSTM_AS_OF_DATE"]) + pd.Timedelta(days=1)).strftime("%Y-%m-%d") if os.environ.get("LSTM_AS_OF_DATE") else "2026-10-14"
AUTO_ADJUST = False
SEQ_LENGTH = 10
TEST_DAYS = 150
VALIDATION_DAYS = 60
MIN_TRAIN_SAMPLES = 80
EPOCHS = 50
BATCH_SIZE = 32
PATIENCE = 5
SEED = 42
PREDICT_DAYS = 30
HISTORY_DAYS = 100
FUTURE_DAYS = 20
WIN_RATE_TEST_DAYS = TEST_DAYS
TOTAL_WIN_TEST_DAYS = TEST_DAYS
HISTORY_METRIC_DAYS = TEST_DAYS
ANNUAL_TRADING_DAYS = 252
RISK_FREE_RATE = 0.0
# 以下是可調整的研究假設。券商費率與折扣應由使用者按實際條件設定。
BROKER_FEE_RATE = .001425
SELL_TAX_RATE = .003
DIRECTION_ZERO_TOL_PCT = 1e-8
LIQUIDATE_AT_END = True
LOCAL_PRICE_CSV = os.environ.get("LSTM_PRICE_CSV") or None  # 可指定本程式輸出的 historical_prices.csv 以離線重跑。
EXTRA_CLOSED_DATES = []  # 可加入颱風臨時休市或交易所公告的額外休市日。
OUTPUT_DIR = Path(os.environ.get("LSTM_OUTPUT_DIR", "lstm_baseline_outputs"))
SAVE_MODELS = os.environ.get("LSTM_SAVE_MODELS", "1") == "1"

# 價格、歷史測試模型、未來推論模型分開快取；避免不同 Section 重訓出不同結果。
_PRICE_CACHE = {}
_TEST_CACHE = {}
_FORECAST_CACHE = {}
_HISTORY_CACHE = {}

## 共用參數檢查與價格資料

先檢查權重與資料格式，再依日期交集合併五檔與大盤。任何一檔資料失敗都會明確停止。

In [ ]:
def set_seeds(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    if tf is not None:
        tf.keras.utils.set_random_seed(seed)

def reset_caches():
    """更改資料或參數後清除快取，再從資料準備步驟重跑。"""
    for cache in (_PRICE_CACHE, _TEST_CACHE, _FORECAST_CACHE, _HISTORY_CACHE):
        cache.clear()

def validate_config():
    weights = np.asarray(list(PORTFOLIO_CONFIG.values()), dtype=float)
    if len(weights) != 5 or not np.isclose(weights.sum(), 1.0) or (weights <= 0).any():
        raise ValueError("舊版對照組必須有五檔股票，且正權重總和為 1。")
    if not np.allclose(weights, .2):
        raise ValueError("構想書的舊版對照組使用五檔各 20%，請保留等權重。")
    if pd.Timestamp(START_DATE_GLOBAL) >= pd.Timestamp(END_DATE_GLOBAL):
        raise ValueError("起始日必須早於截止日。")
    if min(SEQ_LENGTH, TEST_DAYS, VALIDATION_DAYS, MIN_TRAIN_SAMPLES,
           EPOCHS, BATCH_SIZE, PREDICT_DAYS, FUTURE_DAYS) < 1:
        raise ValueError("視窗、期間、訓練次數與批次大小必須為正整數。")
    if INITIAL_CAPITAL <= 0 or RISK_FREE_RATE <= -1:
        raise ValueError("本金必須為正，年化無風險利率必須大於 -1。")
    if not (0 <= BROKER_FEE_RATE < 1 and 0 <= SELL_TAX_RATE < 1
            and BROKER_FEE_RATE + SELL_TAX_RATE < 1):
        raise ValueError("交易成本參數不合法。")

def _fingerprint(frame):
    hashed = pd.util.hash_pandas_object(frame, index=True).values.tobytes()
    return hashlib.sha256(hashed).hexdigest()[:16]

def _direction(return_pct):
    """百分比小於數值容許誤差時視為平盤，避免浮點誤差改變方向。"""
    values = np.asarray(return_pct, dtype=float)
    return np.where(np.abs(values) <= DIRECTION_ZERO_TOL_PCT, 0, np.sign(values)).astype(int)

def _normalise_price_frame(frame, ticker):
    """處理 yfinance 的 MultiIndex 欄位與日期；不使用未來值補資料。"""
    frame = frame.copy()
    if isinstance(frame.columns, pd.MultiIndex):
        frame.columns = frame.columns.get_level_values(0)
    if "Date" in frame.columns:
        frame = frame.set_index("Date")
    if "Close" not in frame.columns:
        raise ValueError(f"{ticker} 的資料缺少 Close 欄位。")
    frame.index = pd.DatetimeIndex(pd.to_datetime(frame.index))
    if frame.index.tz is not None:
        frame.index = frame.index.tz_convert("Asia/Taipei").tz_localize(None)
    frame.index = frame.index.normalize()
    frame.index.name = "Date"
    frame = frame.sort_index()
    if frame.index.duplicated().any():
        raise ValueError(f"{ticker} 有重複日期，請先確認資料來源。")
    frame["Close"] = pd.to_numeric(frame["Close"], errors="coerce")
    frame = frame.dropna(subset=["Close"])
    if frame.empty or not np.isfinite(frame["Close"]).all() or (frame["Close"] <= 0).any():
        raise ValueError(f"{ticker} 無有效資料或出現非正的收盤價。")
    return frame

def prepare_prices(start_date=None, end_date=None):
    """一次讀取五檔及大盤，取共同交易日，將原始價格保存供重現。"""
    start_date = start_date or START_DATE_GLOBAL
    end_date = end_date or END_DATE_GLOBAL
    tickers = list(PORTFOLIO_CONFIG) + [MARKET_TICKER]
    local_path = Path(LOCAL_PRICE_CSV).resolve() if LOCAL_PRICE_CSV else None
    local_stamp = local_path.stat().st_mtime_ns if local_path else None
    key = (start_date, end_date, tuple(tickers), AUTO_ADJUST, str(local_path), local_stamp)
    if key in _PRICE_CACHE:
        return _PRICE_CACHE[key]
    local = None
    if local_path:
        local = pd.read_csv(local_path, dtype={"ticker": str})
        if not {"Date", "ticker", "Close"}.issubset(local.columns):
            raise ValueError("離線 CSV 必須包含 Date、ticker、Close，並提供五檔與 ^TWII。")
    elif yf is None:
        raise ImportError("找不到 yfinance，請先執行 Notebook 的套件安裝儲存格。")
    frames, audit, raw_parts = {}, [], []
    for ticker in tickers:
        if local is not None:
            source = local.loc[local["ticker"] == ticker].drop(columns="ticker")
        else:
            source = yf.download(ticker, start=start_date, end=end_date,
                                 auto_adjust=AUTO_ADJUST, progress=False,
                                 threads=False, timeout=30)
        source = _normalise_price_frame(source, ticker)
        source = source.loc[(source.index >= pd.Timestamp(start_date)) &
                            (source.index < pd.Timestamp(end_date))]
        if source.empty:
            raise ValueError(f"{ticker} 在指定期間沒有資料。")
        frames[ticker] = source
        audit.append({"Ticker": ticker, "Name": STOCK_NAMES.get(ticker, ticker),
                      "Original_Rows": len(source), "First_Date": source.index[0],
                      "Last_Date": source.index[-1]})
        part = source.reset_index()
        part["ticker"] = ticker
        raw_parts.append(part)
    # 使用日期交集；不把六組陣列直接按位置拼在一起，也不靜默略過任何一股。
    panel = pd.concat([frames[t]["Close"].rename(t) for t in tickers], axis=1).dropna()
    panel.index.name = "Date"
    if panel.empty:
        raise ValueError("五檔與大盤沒有共同交易日，無法公平比較。")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    pd.concat(raw_parts, ignore_index=True).to_csv(
        OUTPUT_DIR / "historical_prices.csv", index=False, encoding="utf-8-sig")
    audit_frame = pd.DataFrame(audit)
    audit_frame["Common_Rows"] = len(panel)
    audit_frame.to_csv(OUTPUT_DIR / "data_audit.csv", index=False, encoding="utf-8-sig")
    _PRICE_CACHE[key] = panel
    print(f"共同交易日：{panel.index[0].date()} ～ {panel.index[-1].date()}，{len(panel)} 天")
    display(audit_frame)
    return panel

## LSTM 訓練與未來預測函式

訓練、驗證、測試按標籤日期分開。Scaler 只在訓練資料 fit，EarlyStopping 監控 val_loss 並恢復最佳權重。

歷史測試固定模型，逐日使用截至前一交易日已知的 10 日收盤價預測下一日；這與「一次預測未來 150 天」不同。

未來模型另以全部已知歷史重新訓練，訓練次數採歷史驗證集選出的最佳 epoch；不將該模型拿回歷史測試計算勝率。補齊 get_history_and_forecast，供時光機使用。

In [ ]:
def build_lstm_model(n_features=1):
    """沿用原始雙層 50 單元 LSTM；舊版實際使用 (10, 1)。"""
    if tf is None:
        raise ImportError("找不到 TensorFlow，請先執行套件安裝儲存格。")
    model = Sequential([Input(shape=(SEQ_LENGTH, n_features)),
                        LSTM(50, return_sequences=True), Dropout(.2),
                        LSTM(50), Dropout(.2), Dense(1)])
    model.compile(optimizer="adam", loss="mean_squared_error")
    return model

def prepare_univariate_windows(prices, test_days, validation_days=None):
    """依標籤日期切分。Scaler 僅學習訓練期間；測試使用已知歷史做逐日一步預測。"""
    validation_days = VALIDATION_DAYS if validation_days is None else validation_days
    prices = pd.Series(prices).astype(float).sort_index()
    if prices.index.duplicated().any() or prices.isna().any():
        raise ValueError("建立序列前必須完成去重與缺失值檢查。")
    if not np.isfinite(prices).all() or (prices <= 0).any():
        raise ValueError("收盤價必須為有限的正數。")
    n = len(prices)
    test_start = n - test_days
    train_end = test_start - validation_days
    training_samples = train_end - SEQ_LENGTH
    if min(test_days, validation_days) < 1 or training_samples < MIN_TRAIN_SAMPLES:
        required = SEQ_LENGTH + MIN_TRAIN_SAMPLES + validation_days + test_days
        raise ValueError(f"只有 {n} 天資料，此設定至少需要 {required} 天。"
                         "請延長歷史期間，或明確調整測試/驗證天數。")
    scaler = MinMaxScaler((0, 1))
    # train_end 不包含在訓練期間內；最後一個訓練答案是 train_end - 1。
    scaler.fit(prices.iloc[:train_end].to_numpy().reshape(-1, 1))
    scaled = scaler.transform(prices.to_numpy().reshape(-1, 1))
    label_indices = np.arange(SEQ_LENGTH, n)
    X = np.stack([scaled[i-SEQ_LENGTH:i] for i in label_indices])
    y = scaled[label_indices]
    train_mask = label_indices < train_end
    val_mask = (label_indices >= train_end) & (label_indices < test_start)
    test_mask = label_indices >= test_start
    return {"X_train": X[train_mask], "y_train": y[train_mask],
            "X_val": X[val_mask], "y_val": y[val_mask],
            "X_test": X[test_mask], "y_test": y[test_mask],
            "train_dates": prices.index[label_indices[train_mask]],
            "val_dates": prices.index[label_indices[val_mask]],
            "test_dates": prices.index[label_indices[test_mask]],
            "previous_close": prices.iloc[test_start-1:n-1].to_numpy(),
            "scaler": scaler, "scaled_all": scaled,
            "train_end": train_end, "test_start": test_start}

def _training_key(ticker, prices, test_days):
    return (ticker, _fingerprint(prices), test_days, VALIDATION_DAYS, SEQ_LENGTH,
            MIN_TRAIN_SAMPLES, EPOCHS, BATCH_SIZE, PATIENCE, SEED)

def _save_model_bundle(bundle, ticker, purpose):
    if not SAVE_MODELS:
        return
    folder = OUTPUT_DIR / "models"
    folder.mkdir(parents=True, exist_ok=True)
    safe_ticker = ticker.replace("^", "").replace(".", "_")
    bundle["model"].save(folder / f"{safe_ticker}_{purpose}.keras")
    joblib.dump(bundle["scaler"], folder / f"{safe_ticker}_{purpose}_scaler.joblib")

def get_test_bundle(ticker, test_days=TEST_DAYS, start_date=None, end_date=None):
    panel = prepare_prices(start_date, end_date)
    prices = panel[ticker]
    key = _training_key(ticker, prices, test_days)
    if key in _TEST_CACHE:
        return _TEST_CACHE[key]
    data = prepare_univariate_windows(prices, test_days)
    print(f"訓練 {ticker}：train={len(data['X_train'])}, val={len(data['X_val'])}, test={len(data['X_test'])}")
    set_seeds()
    model = build_lstm_model()
    callback = EarlyStopping(monitor="val_loss", patience=PATIENCE,
                             restore_best_weights=True, mode="min")
    history = model.fit(data["X_train"], data["y_train"],
                        validation_data=(data["X_val"], data["y_val"]),
                        epochs=EPOCHS, batch_size=BATCH_SIZE, shuffle=False,
                        callbacks=[callback], verbose=0)
    losses = np.asarray(history.history["val_loss"], dtype=float)
    if not np.isfinite(losses).all():
        raise RuntimeError(f"{ticker} 的驗證損失出現 NaN/Inf，請檢查資料。")
    predicted = data["scaler"].inverse_transform(
        model.predict(data["X_test"], verbose=0)).ravel()
    actual = prices.loc[data["test_dates"]].to_numpy(dtype=float)
    if not np.isfinite(predicted).all() or (predicted <= 0).any():
        raise RuntimeError(f"{ticker} 預測出非有限或非正價格，不能以截斷值掩蓋問題。")
    result = pd.DataFrame({"Date": data["test_dates"], "Ticker": ticker,
                           "Signal_Date": prices.index[data["test_start"]-1:-1],
                           "Previous_Close": data["previous_close"],
                           "Actual_Close": actual, "Predicted_Close": predicted})
    result["Actual_Return_Pct"] = (result.Actual_Close/result.Previous_Close - 1)*100
    result["Predicted_Return_Pct"] = (result.Predicted_Close/result.Previous_Close - 1)*100
    result["Actual_Direction"] = _direction(result.Actual_Return_Pct)
    result["Predicted_Direction"] = _direction(result.Predicted_Return_Pct)
    result["Direction_Correct"] = result.Actual_Direction.eq(result.Predicted_Direction)
    result["Rolling_Accuracy_Pct"] = result.Direction_Correct.rolling(20).mean()*100
    split = {"Ticker": ticker, "Train_Start": str(data["train_dates"][0].date()),
             "Train_End": str(data["train_dates"][-1].date()),
             "Validation_Start": str(data["val_dates"][0].date()),
             "Validation_End": str(data["val_dates"][-1].date()),
             "Test_Start": str(data["test_dates"][0].date()),
             "Test_End": str(data["test_dates"][-1].date()),
             "Train_Samples": len(data["X_train"]), "Validation_Samples": len(data["X_val"]),
             "Test_Samples": len(data["X_test"]), "Best_Epoch": int(losses.argmin()+1)}
    bundle = {"model": model, "scaler": data["scaler"], "prices": prices,
              "predictions": result, "split": split, "history": history.history,
              "best_epoch": split["Best_Epoch"], "data_key": key}
    _TEST_CACHE[key] = bundle
    _save_model_bundle(bundle, ticker, "backtest")
    return bundle

def train_and_predict_test_set(ticker, test_days, start_date=None, end_date=None):
    """保留原始介面；其他 Section 改為重用快取。"""
    b = get_test_bundle(ticker, test_days, start_date, end_date)
    p = b["predictions"]
    return pd.DatetimeIndex(p.Date), p.Actual_Close.to_numpy(), p.Predicted_Close.to_numpy()

def get_forecast_bundle(ticker, start_date=None, end_date=None):
    """未來推論另用已知全部歷史重訓，與未見測試集的評估模型分開。"""
    panel = prepare_prices(start_date, end_date)
    prices = panel[ticker]
    base = get_test_bundle(ticker, TEST_DAYS, start_date, end_date)
    key = (base["data_key"], base["best_epoch"], "future")
    if key in _FORECAST_CACHE:
        return _FORECAST_CACHE[key]
    scaler = MinMaxScaler((0, 1))
    scaled = scaler.fit_transform(prices.to_numpy().reshape(-1, 1))
    indices = np.arange(SEQ_LENGTH, len(prices))
    X = np.stack([scaled[i-SEQ_LENGTH:i] for i in indices])
    y = scaled[indices]
    set_seeds()
    model = build_lstm_model()
    print(f"未來推論模型重訓 {ticker}：使用全部已知歷史，epochs={base['best_epoch']}")
    model.fit(X, y, epochs=base["best_epoch"], batch_size=BATCH_SIZE,
              shuffle=False, verbose=0)
    bundle = {"model": model, "scaler": scaler, "prices": prices, "scaled": scaled}
    _FORECAST_CACHE[key] = bundle
    _save_model_bundle(bundle, ticker, "forecast")
    return bundle

def predict_price_sequence(ticker, days_to_predict, start_date=None, end_date=None):
    """原始單變量模型可將預測價格遞迴輸入；不是已知未來的真實股價。"""
    if days_to_predict < 1:
        raise ValueError("預測天數必須為正整數。")
    b = get_forecast_bundle(ticker, start_date, end_date)
    window = b["scaled"][-SEQ_LENGTH:].reshape(1, SEQ_LENGTH, 1)
    output = [float(b["prices"].iloc[-1])]
    for _ in range(days_to_predict):
        scaled_prediction = np.asarray(b["model"].predict(window, verbose=0)).reshape(1, 1)
        value = float(b["scaler"].inverse_transform(scaled_prediction)[0, 0])
        if not np.isfinite(value) or value <= 0:
            raise RuntimeError(f"{ticker} 遞迴預測產生無效價格，請檢查模型。")
        output.append(value)
        window = np.concatenate([window[:, 1:, :], scaled_prediction.reshape(1, 1, 1)], axis=1)
    return b["prices"].index[-1], output

def future_trading_dates(last_date, days):
    """用 XTAI 台灣交易日曆排除休市日，並允許補入臨時休市日。"""
    if xcals is None:
        raise ImportError("未安裝 exchange-calendars，請先執行套件安裝儲存格。")
    start = pd.Timestamp(last_date).normalize() + pd.Timedelta(days=1)
    end = start + pd.Timedelta(days=days*4 + 60)
    calendar = xcals.get_calendar("XTAI", start=start-pd.Timedelta(days=7), end=end)
    dates = pd.DatetimeIndex(calendar.sessions_in_range(start, end))
    if dates.tz is not None:
        dates = dates.tz_localize(None)
    dates = dates.normalize().difference(pd.DatetimeIndex(pd.to_datetime(EXTRA_CLOSED_DATES)))
    if len(dates) < days:
        raise ValueError("台股交易日曆不足以建立所需的未來日期。")
    return dates[:days]

def get_history_and_forecast(ticker, history_days, future_days, start_date=None, end_date=None):
    """補回原始時光機缺失的函式；回傳 (歷史日期, 價格)、(未來日期, 價格)。"""
    prices = prepare_prices(start_date, end_date)[ticker]
    if history_days < 2 or len(prices) < history_days:
        raise ValueError("時光機歷史天數不足或設定少於 2 天。")
    last_date, predictions = predict_price_sequence(ticker, future_days, start_date, end_date)
    past = prices.iloc[-history_days:]
    return ((past.index, past.to_numpy()),
            (future_trading_dates(last_date, future_days), np.asarray(predictions[1:])))

## 回測績效與方向檢驗函式

MDD 使用負百分比表示，例如 -25 代表自高點最大回落 25%。

固定等權重表示買入時各配置 20%，之後固定股數持有，權重會隨價格變動；未執行每日再平衡。每筆配置先預留買入手續費，回測期末依設定扣掉賣出手續費與交易稅。

第一個淨值點為買入前本金，首次持有期報酬包含買入成本。採可分割股數的研究近似，尚未加入股數取整、最低手續費、滑價與股息再投資。

方向準確率由預測與真實價格相對前一日真實收盤價計算，保留第一個測試日。平盤容許誤差以百分比 1e-8 定義，並分開列出固定猜上漲基準。

In [ ]:
def _valid_values(values):
    arr = np.asarray(values, dtype=float).reshape(-1)
    if len(arr) < 2 or not np.isfinite(arr).all() or (arr <= 0).any():
        raise ValueError("資產價值需至少兩筆有限的正數。")
    return arr

def calculate_sharpe(values, risk_free_rate=None, annual_days=None):
    arr = _valid_values(values)
    risk_free_rate = RISK_FREE_RATE if risk_free_rate is None else risk_free_rate
    annual_days = ANNUAL_TRADING_DAYS if annual_days is None else annual_days
    returns = arr[1:]/arr[:-1] - 1
    daily_rf = (1+risk_free_rate)**(1/annual_days) - 1
    volatility = returns.std(ddof=1) if len(returns) > 1 else 0.0
    return 0.0 if volatility < 1e-12 else float((returns.mean()-daily_rf)/volatility*np.sqrt(annual_days))

def calculate_max_drawdown(values):
    """以負百分比表示最大回撤，例如 -20 代表曾從高點回落 20%。"""
    arr = _valid_values(values)
    return float((arr/np.maximum.accumulate(arr)-1).min()*100)

def calculate_pearson_correlation(values_a, values_b):
    a = pd.Series(values_a).pct_change(fill_method=None)
    b = pd.Series(values_b).pct_change(fill_method=None)
    pair = pd.concat([a.rename("A"), b.rename("B")], axis=1).dropna()
    if len(pair) < 2 or pair.A.std() < 1e-12 or pair.B.std() < 1e-12:
        return np.nan
    return float(pair.A.corr(pair.B))

def equal_weight_nav(prices, initial_capital=None, charge_costs=True, liquidate=None):
    """固定股數持有，不每日再平衡。第一列是買進前本金；成本在回測期間內計入。"""
    capital = INITIAL_CAPITAL if initial_capital is None else initial_capital
    liquidate = LIQUIDATE_AT_END if liquidate is None else liquidate
    tickers = list(PORTFOLIO_CONFIG)
    selected = prices.loc[:, tickers].astype(float)
    if len(selected) < 2 or selected.isna().any().any() or (selected <= 0).any().any():
        raise ValueError("固定配置回測需要五檔完整且正值的價格及至少兩個日期。")
    if not np.isfinite(selected.to_numpy()).all() or capital <= 0:
        raise ValueError("價格與本金不合法。")
    fee = BROKER_FEE_RATE if charge_costs else 0.0
    weights = pd.Series(PORTFOLIO_CONFIG)
    shares = capital*weights/(selected.iloc[0]*(1+fee))
    values = selected.mul(shares, axis=1).sum(axis=1)
    buy_fee = float((shares*selected.iloc[0]).sum()*fee)
    sale_fee = float(values.iloc[-1]*fee) if charge_costs and liquidate else 0.0
    sale_tax = float(values.iloc[-1]*SELL_TAX_RATE) if charge_costs and liquidate else 0.0
    values.iloc[-1] -= sale_fee + sale_tax
    # 首列保留買進前本金，讓第一個持有期報酬納入買進費用，MDD 不漏掉成本。
    values.iloc[0] = capital
    ledger = {"Buy_Fee": buy_fee, "Sell_Fee": sale_fee, "Sell_Tax": sale_tax,
              "Total_Cost": buy_fee+sale_fee+sale_tax,
              "Buy_Orders": len(tickers), "Sell_Orders": len(tickers) if liquidate else 0}
    return values, shares, ledger

def calculate_portfolio_value_generic(portfolio_data, initial_capital, real_price_key,
                                      pred_price_key=None, mkt_real=None):
    """相容原始介面的無成本價格推估；正式成本績效使用 equal_weight_nav。"""
    tickers = list(PORTFOLIO_CONFIG)
    real = pd.DataFrame({t: np.asarray(portfolio_data[real_price_key][t]).reshape(-1) for t in tickers})
    real_nav, _, _ = equal_weight_nav(real, initial_capital, charge_costs=False, liquidate=False)
    predicted_nav = None
    if pred_price_key:
        pred = pd.DataFrame({t: np.asarray(portfolio_data[pred_price_key][t]).reshape(-1) for t in tickers})
        shares = initial_capital*pd.Series(PORTFOLIO_CONFIG)/real.iloc[0]
        predicted_nav = pred.mul(shares, axis=1).sum(axis=1).to_numpy()
    market_nav = None if mkt_real is None else initial_capital*np.asarray(mkt_real)/float(np.asarray(mkt_real)[0])
    return (100*(real_nav.to_numpy()/initial_capital-1),
            None if predicted_nav is None else 100*(predicted_nav/initial_capital-1),
            None if market_nav is None else 100*(market_nav/initial_capital-1),
            real_nav.to_numpy(), market_nav)

def get_history_bundle(test_days=None):
    """六個模型使用同一測試日期；所有歷史 Section 共用此結果。"""
    test_days = TEST_DAYS if test_days is None else test_days
    panel = prepare_prices()
    key = (_fingerprint(panel), tuple(PORTFOLIO_CONFIG), test_days, VALIDATION_DAYS, SEQ_LENGTH, EPOCHS,
           BATCH_SIZE, PATIENCE, SEED, INITIAL_CAPITAL, BROKER_FEE_RATE,
           SELL_TAX_RATE, LIQUIDATE_AT_END)
    if key in _HISTORY_CACHE:
        return _HISTORY_CACHE[key]
    bundles = {t: get_test_bundle(t, test_days) for t in panel.columns}
    actual_prices = panel.iloc[-test_days-1:]
    predicted_prices = actual_prices.copy()
    for t in panel.columns:
        p = bundles[t]["predictions"]
        if not pd.DatetimeIndex(p.Date).equals(actual_prices.index[1:]):
            raise ValueError(f"{t} 的測試日期不一致，不能合成投資組合。")
        predicted_prices.loc[actual_prices.index[1:], t] = p.Predicted_Close.to_numpy()
    gross_nav, _, _ = equal_weight_nav(actual_prices, charge_costs=False, liquidate=False)
    net_nav, _, costs = equal_weight_nav(actual_prices, charge_costs=True)
    predicted_nav, _, _ = equal_weight_nav(predicted_prices, charge_costs=True)
    market_nav = INITIAL_CAPITAL*actual_prices[MARKET_TICKER]/actual_prices[MARKET_TICKER].iloc[0]
    shares_gross = INITIAL_CAPITAL*pd.Series(PORTFOLIO_CONFIG)/actual_prices.iloc[0][list(PORTFOLIO_CONFIG)]
    predicted_gross = predicted_prices[list(PORTFOLIO_CONFIG)].mul(shares_gross, axis=1).sum(axis=1)
    # 方向評估用固定組合的價格變動，不把一次性手續費當成模型判斷的漲跌。
    previous = gross_nav.iloc[:-1].to_numpy()
    actual_change = gross_nav.iloc[1:].to_numpy()-previous
    predicted_change = predicted_gross.iloc[1:].to_numpy()-previous
    direction = pd.DataFrame({"Date": actual_prices.index[1:],
                              "Actual_Direction": _direction(actual_change/previous*100),
                              "Predicted_Direction": _direction(predicted_change/previous*100)})
    direction["Direction_Correct"] = direction.Actual_Direction.eq(direction.Predicted_Direction)
    direction["Rolling_Accuracy_Pct"] = direction.Direction_Correct.rolling(20).mean()*100
    nav = pd.DataFrame({"Portfolio_Gross": gross_nav, "Portfolio_Net": net_nav,
                        "Predicted_Portfolio_Value": predicted_nav, "Market_Index": market_nav})
    nav["Portfolio_Net_Drawdown_Pct"] = 100*(net_nav/net_nav.cummax()-1)
    nav["Market_Drawdown_Pct"] = 100*(market_nav/market_nav.cummax()-1)
    result = {"models": bundles, "nav": nav, "portfolio_direction": direction, "costs": costs}
    _HISTORY_CACHE[key] = result
    return result

def calculate_win_rate(ticker, test_days, start_date=None, end_date=None):
    b = get_test_bundle(ticker, test_days, start_date, end_date)
    p = b["predictions"].copy()
    p = p.rename(columns={"Actual_Close": "Real Price", "Predicted_Close": "Pred Price"})
    p["Win"] = p.Direction_Correct.astype(int)
    return float(p.Win.mean()*100), p

def _accuracy_plot(frame, title):
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(frame.Date, frame.Rolling_Accuracy_Pct, label="20-day direction accuracy")
    ax.axhline(50, color="grey", linestyle="--", label="50% reference")
    ax.set(title=title, ylabel="Accuracy (%)", xlabel="Target date", ylim=(0, 100))
    ax.grid(alpha=.25)
    ax.legend()
    fig.tight_layout()
    plt.show()

## 原始八個分析段落與匯出函式

這些函式保留原始 Section 名稱。每個段落共用相同日期、相同模型預測及相同資產淨值。

固定持有 ROI 與 Sharpe 衡量的是固定持有策略；價格誤差與方向準確率才用來檢驗 LSTM。不能將固定配置跑贏大盤解讀為模型已經產生交易獲利。

In [ ]:
def run_section_1_future_prediction():
    forecasts, last_date = {}, None
    for t in list(PORTFOLIO_CONFIG)+[MARKET_TICKER]:
        date, sequence = predict_price_sequence(t, PREDICT_DAYS)
        forecasts[t] = sequence
        if last_date is not None and date != last_date:
            raise ValueError("各標的預測基準日不一致。")
        last_date = date
    dates = pd.DatetimeIndex([last_date]).append(future_trading_dates(last_date, PREDICT_DAYS))
    prices = pd.DataFrame(forecasts, index=dates)
    pf, _, _ = equal_weight_nav(prices, charge_costs=False, liquidate=False)
    mkt = INITIAL_CAPITAL*prices[MARKET_TICKER]/prices[MARKET_TICKER].iloc[0]
    table = prices.copy()
    table["Portfolio_Forecast_ROI_Pct"] = 100*(pf/INITIAL_CAPITAL-1)
    table["Market_Forecast_ROI_Pct"] = 100*(mkt/INITIAL_CAPITAL-1)
    table.index.name = "Date"
    display(table)
    table.to_csv(OUTPUT_DIR/"future_30_trading_days.csv", encoding="utf-8-sig")
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.plot(dates, table.Portfolio_Forecast_ROI_Pct, label="Portfolio forecast")
    ax.plot(dates, table.Market_Forecast_ROI_Pct, label="Market forecast", linestyle="--")
    ax.axhline(0, color="grey", linewidth=.8)
    ax.set(title=f"Recursive forecast from {last_date.date()} ({PREDICT_DAYS} trading days)",
           ylabel="Forecast ROI (%)", xlabel="Date")
    ax.grid(alpha=.25); ax.legend(); fig.tight_layout(); plt.show()
    print("這是未來情境推估；尚未發生的預測結果不列入歷史 ROI、Sharpe 或勝率。")
    return table

def run_section_2_backtest():
    b = get_history_bundle()
    nav = b["nav"]
    fig, axes = plt.subplots(3, 1, figsize=(12, 11), sharex=True)
    roi = 100*(nav[["Portfolio_Gross", "Portfolio_Net", "Market_Index"]]/INITIAL_CAPITAL-1)
    # 全部使用 Matplotlib 日期座標，避免 Pandas Period 座標與 datetime 混用而讓曲線消失。
    for column in roi.columns:
        axes[0].plot(roi.index, roi[column].to_numpy(), label=column)
    axes[0].legend(); axes[0].set_ylabel("ROI (%)")
    axes[0].set_title(f"Fixed holdings backtest: {TEST_DAYS} held-out trading days")
    for t in PORTFOLIO_CONFIG:
        p = b["models"][t]["predictions"]
        axes[1].plot(p.Date, p.Predicted_Return_Pct-p.Actual_Return_Pct, alpha=.7, label=t)
    axes[1].axhline(0, color="grey", linewidth=.8)
    axes[1].set_ylabel("Return error (pp)"); axes[1].legend()
    for column in ["Portfolio_Net_Drawdown_Pct", "Market_Drawdown_Pct"]:
        axes[2].plot(nav.index, nav[column].to_numpy(), label=column)
    axes[2].legend()
    axes[2].set(ylabel="Drawdown (%)", xlabel="Date")
    for ax in axes: ax.grid(alpha=.25)
    fig.tight_layout(); plt.show()
    print(f"測試目標日期：{nav.index[1].date()} ～ {nav.index[-1].date()}，共 {TEST_DAYS} 天")
    print("固定持有對照組的實際績效與模型預測誤差分開呈現。")
    display(pd.DataFrame([b["costs"]]))
    return nav

def model_metric_table():
    b = get_history_bundle()
    rows = []
    for t, bundle in b["models"].items():
        p = bundle["predictions"]
        rows.append({"Ticker": t, "Name": STOCK_NAMES.get(t, t), "Test_Days": len(p),
                     "Price_MAE": mean_absolute_error(p.Actual_Close, p.Predicted_Close),
                     "Price_RMSE": np.sqrt(mean_squared_error(p.Actual_Close, p.Predicted_Close)),
                     "Naive_Last_Close_MAE": mean_absolute_error(p.Actual_Close, p.Previous_Close),
                     "Return_MAE_pp": mean_absolute_error(p.Actual_Return_Pct, p.Predicted_Return_Pct),
                     "Return_RMSE_pp": np.sqrt(mean_squared_error(p.Actual_Return_Pct, p.Predicted_Return_Pct)),
                     "Direction_Accuracy_Pct": p.Direction_Correct.mean()*100,
                     "Always_Up_Accuracy_Pct": p.Actual_Direction.eq(1).mean()*100})
    return pd.DataFrame(rows)

def run_section_3_win_rate():
    b = get_history_bundle()
    table = model_metric_table()
    display(table.loc[table.Ticker.isin(PORTFOLIO_CONFIG)])
    for t in PORTFOLIO_CONFIG:
        _accuracy_plot(b["models"][t]["predictions"], f"{t}: held-out direction accuracy")
    print("方向準確率與交易勝率不同；請與固定猜上漲基準及樣本量一起解讀。")
    return table

def run_section_4_time_machine():
    parts = {t: get_history_and_forecast(t, HISTORY_DAYS, FUTURE_DAYS)
             for t in list(PORTFOLIO_CONFIG)+[MARKET_TICKER]}
    past_dates, _ = parts[MARKET_TICKER][0]
    future_dates, _ = parts[MARKET_TICKER][1]
    past_prices = pd.DataFrame({t: part[0][1] for t, part in parts.items()}, index=past_dates)
    future_prices = pd.DataFrame({t: part[1][1] for t, part in parts.items()}, index=future_dates)
    past_pf, shares, _ = equal_weight_nav(past_prices, charge_costs=False, liquidate=False)
    future_pf = future_prices[list(PORTFOLIO_CONFIG)].mul(shares, axis=1).sum(axis=1)
    market_shares = INITIAL_CAPITAL/past_prices[MARKET_TICKER].iloc[0]
    past_market = market_shares*past_prices[MARKET_TICKER]
    future_market = market_shares*future_prices[MARKET_TICKER]
    fig, ax = plt.subplots(figsize=(12, 5))
    for label, actual, forecast in [("Portfolio", past_pf, future_pf),
                                     ("Market", past_market, future_market)]:
        line, = ax.plot(actual.index, 100*(actual/INITIAL_CAPITAL-1), label=f"{label} actual")
        connection = pd.concat([actual.iloc[-1:], forecast])
        ax.plot(connection.index, 100*(connection/INITIAL_CAPITAL-1),
                linestyle="--", color=line.get_color(), label=f"{label} forecast")
    ax.axvline(past_dates[-1], color="grey", linestyle=":")
    ax.set(title="History and forecast (gross fixed holdings)", ylabel="ROI (%)", xlabel="Date")
    ax.grid(alpha=.25); ax.legend(); fig.tight_layout(); plt.show()
    combined = pd.DataFrame({"Portfolio_Value": pd.concat([past_pf, future_pf]),
                             "Market_Value": pd.concat([past_market, future_market])})
    combined["Segment"] = ["Actual"]*len(past_dates)+["Forecast"]*len(future_dates)
    combined.index.name = "Date"
    combined.to_csv(OUTPUT_DIR/"time_machine.csv", encoding="utf-8-sig")
    print("時光機顯示未扣成本的歷史與情境延伸；正式含成本績效使用歷史回測總表。")
    return combined

def run_section_5_portfolio_win_rate():
    p = get_history_bundle()["portfolio_direction"]
    display(pd.DataFrame([{"Test_Days": len(p),
                           "Portfolio_Direction_Accuracy_Pct": p.Direction_Correct.mean()*100,
                           "Always_Up_Accuracy_Pct": p.Actual_Direction.eq(1).mean()*100}]))
    _accuracy_plot(p, "Portfolio: held-out direction accuracy")
    return p

def _compute_history_metrics_for_sections(test_days):
    b = get_history_bundle(test_days)
    nav = b["nav"]
    return (calculate_sharpe(nav.Portfolio_Net), calculate_sharpe(nav.Market_Index),
            calculate_pearson_correlation(nav.Portfolio_Net, nav.Market_Index), nav.index)

def run_section_6_portfolio_sharpe():
    nav = get_history_bundle()["nav"]
    table = pd.DataFrame([{"Strategy": "Old fixed equal-weight (net)",
                           "Sharpe": calculate_sharpe(nav.Portfolio_Net),
                           "MDD_Pct": calculate_max_drawdown(nav.Portfolio_Net)}])
    display(table)
    return table

def run_section_7_market_sharpe():
    nav = get_history_bundle()["nav"]
    table = pd.DataFrame([{"Strategy": "TWII price-index benchmark", "Sharpe": calculate_sharpe(nav.Market_Index),
                           "MDD_Pct": calculate_max_drawdown(nav.Market_Index)}])
    display(table)
    return table

def run_section_8_portfolio_market_corr():
    nav = get_history_bundle()["nav"]
    correlation = calculate_pearson_correlation(nav.Portfolio_Net, nav.Market_Index)
    display(pd.DataFrame([{"Daily_Return_Pearson_Correlation": correlation}]))
    print("相關係數只衡量線性關係；接近零不代表兩者在統計上獨立。")
    return correlation

def market_risk_features(panel=None):
    """輸出日後風控可用的欄位；此舊版對照組仍保持固定權重。"""
    panel = prepare_prices() if panel is None else panel
    risk = pd.DataFrame(index=panel.index)
    risk["Market_Close"] = panel[MARKET_TICKER]
    risk["Market_Return"] = risk.Market_Close.pct_change(fill_method=None)
    risk["Market_Vol_20d"] = risk.Market_Return.rolling(20).std(ddof=1)*np.sqrt(252)*100
    # 閾值只能來自之前已觀察的波動率，不能由完整回測期間計算後倒套回去。
    previous_vol = risk.Market_Vol_20d.shift(1)
    risk["Risk_Threshold_Pct"] = (previous_vol.expanding(min_periods=60).mean()
                                  + previous_vol.expanding(min_periods=60).std(ddof=1))
    risk["Risk_Off"] = risk.Market_Vol_20d.gt(risk.Risk_Threshold_Pct).astype("boolean")
    risk.loc[risk.Risk_Threshold_Pct.isna(), "Risk_Off"] = pd.NA
    return risk

def export_all_results():
    """輸出構想書三方總表格式；新版尚未串接的結果保持缺值，絕不填入預期成效。"""
    b = get_history_bundle()
    nav = b["nav"]
    accuracy = b["portfolio_direction"].Direction_Correct.mean()*100
    records = []
    for name, column, acc, status in [
        ("基準大盤 ^TWII", "Market_Index", np.nan, "完成價格指數比較"),
        ("舊版單變量 LSTM＋固定等權重", "Portfolio_Net", accuracy, "完成；ROI 為固定持有績效")]:
        series = nav[column]
        records.append({"Strategy": name, "ROI_Pct": (series.iloc[-1]/INITIAL_CAPITAL-1)*100,
                        "Sharpe": calculate_sharpe(series), "MDD_Pct": calculate_max_drawdown(series),
                        "Direction_Accuracy_Pct": acc, "Test_Days": TEST_DAYS,
                        "Test_Start": nav.index[1].date().isoformat(),
                        "Test_End": nav.index[-1].date().isoformat(), "Status": status})
    records.append({"Strategy": "新版多變量＋新聞情緒＋動態風控", "ROI_Pct": np.nan,
                    "Sharpe": np.nan, "MDD_Pct": np.nan, "Direction_Accuracy_Pct": np.nan,
                    "Test_Days": np.nan, "Test_Start": "", "Test_End": "",
                    "Status": "待後續串接；未進行新版實驗"})
    comparison = pd.DataFrame(records)
    display(comparison)
    predictions = pd.concat([x["predictions"] for x in b["models"].values()], ignore_index=True)
    splits = pd.DataFrame([x["split"] for x in b["models"].values()])
    tables = {"strategy_comparison.csv": comparison,
              "model_metrics.csv": model_metric_table(), "test_predictions.csv": predictions,
              "split_audit.csv": splits, "portfolio_direction.csv": b["portfolio_direction"],
              "transaction_costs.csv": pd.DataFrame([b["costs"]])}
    for filename, table in tables.items():
        table.to_csv(OUTPUT_DIR/filename, index=False, encoding="utf-8-sig")
    nav.to_csv(OUTPUT_DIR/"backtest_nav.csv", encoding="utf-8-sig")
    market_risk_features().to_csv(OUTPUT_DIR/"market_risk_features.csv", encoding="utf-8-sig")
    for t, bundle in b["models"].items():
        training_history = pd.DataFrame(bundle["history"])
        training_history.index = np.arange(1, len(training_history)+1)
        training_history.rename_axis("Epoch").to_csv(
            OUTPUT_DIR/f"training_history_{t.replace('^','').replace('.','_')}.csv", encoding="utf-8-sig")
    config = {"portfolio_weights": PORTFOLIO_CONFIG, "start": START_DATE_GLOBAL,
              "end_exclusive": END_DATE_GLOBAL, "auto_adjust": AUTO_ADJUST,
              "sequence_length": SEQ_LENGTH, "test_days": TEST_DAYS,
              "validation_days": VALIDATION_DAYS, "min_train_samples": MIN_TRAIN_SAMPLES,
              "epochs_max": EPOCHS, "batch_size": BATCH_SIZE, "patience": PATIENCE,
              "seed": SEED, "initial_capital": INITIAL_CAPITAL,
              "direction_zero_tolerance_pct": DIRECTION_ZERO_TOL_PCT,
              "broker_fee_rate": BROKER_FEE_RATE, "sell_tax_rate": SELL_TAX_RATE,
              "liquidate_at_end": LIQUIDATE_AT_END,
              "latest_common_date": nav.index[-1].date().isoformat(),
              "extra_closed_dates": [str(x) for x in EXTRA_CLOSED_DATES],
              "model_target": "Next trading day's Close; price units",
              "test_protocol": "chronological fixed model, rolling one-step evaluation",
              "forecast_protocol": "separate model refit on all observed prices",
              "portfolio_protocol": "fixed shares; fractional-share research approximation",
              "dividends": "not reinvested when auto_adjust is False",
              "benchmark": "TWII theoretical price index; no broker fee or sell tax"}
    (OUTPUT_DIR/"run_config.json").write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"輸出完成：{OUTPUT_DIR.resolve()}")
    display(pd.DataFrame({"File": [str(p.relative_to(OUTPUT_DIR)) for p in sorted(OUTPUT_DIR.rglob('*')) if p.is_file()]}))
    return comparison

def validate_core_calculations():
    """輕量核心驗收：手算 MDD、平盤成本、時序切分與 scaler 不洩漏。"""
    assert np.isclose(calculate_max_drawdown([100, 120, 90, 110]), -25.0)
    assert np.isclose(calculate_max_drawdown([100, 100, 100]), 0.0)
    assert np.isclose(calculate_sharpe([100, 100, 100]), 0.0)
    constant = pd.DataFrame(100.0, index=pd.bdate_range("2020-01-01", periods=3), columns=list(PORTFOLIO_CONFIG))
    nav, _, costs = equal_weight_nav(constant, initial_capital=100000, liquidate=True)
    expected = 100000/(1+BROKER_FEE_RATE)*(1-BROKER_FEE_RATE-SELL_TAX_RATE)
    assert np.isclose(nav.iloc[-1], expected)
    assert np.isclose(100000-nav.iloc[-1], costs["Total_Cost"])
    gross, _, _ = equal_weight_nav(constant, charge_costs=False, liquidate=False)
    assert np.allclose(gross, 100000)
    length = SEQ_LENGTH+MIN_TRAIN_SAMPLES+VALIDATION_DAYS+TEST_DAYS+20
    series = pd.Series(np.linspace(100, 200, length), index=pd.bdate_range("2010-01-01", periods=length))
    data = prepare_univariate_windows(series, TEST_DAYS)
    assert data["train_dates"][-1] < data["val_dates"][0] < data["test_dates"][0]
    assert len(data["X_test"]) == TEST_DAYS
    assert np.isclose(data["scaler"].data_max_[0], series.iloc[data["train_end"]-1])
    assert data["X_test"].shape[1:] == (SEQ_LENGTH, 1)
    print("核心驗收通過：MDD、成本、時間切分、縮放器與模型輸入形狀。")
    return True

## 執行前核心驗收

先用可手算的資料檢查 MDD、平盤交易成本、時間切分與 Scaler。
這是程式計算驗收，與後面的真實模型實驗結果分開。

In [ ]:
validate_config()
validate_core_calculations()
price_panel = prepare_prices()

## 模型訓練與切分檢查

此步驟只訓練歷史測試模型。訓練日期早於驗證日期，驗證日期早於測試日期。
六個標的使用相同交易日，再將分割表顯示供報告核對。
已完成的模型會保存至輸出資料夾的 models 子資料夾。

In [ ]:
historical_results = get_history_bundle()
split_audit = pd.DataFrame([b["split"] for b in historical_results["models"].values()])
display(split_audit)

## 1 未來 30 個交易日預測

另訓練未來推論模型後，將單變量價格預測遞迴輸入。此圖是未來情境推估，不會將尚未發生的預測 ROI 當作回測績效。

In [ ]:
run_section_1_future_prediction()

## 2 歷史 150 個交易日回測

列出固定持有的稅費前、稅費後淨值與 ^TWII 基準，以及模型報酬率預測誤差和回撤。大盤為理論價格指數，不是可直接成交的 ETF，未對指數收取手續費或交易稅。

In [ ]:
run_section_2_backtest()

## 3 每檔股票方向準確率

表格包含價格 MAE/RMSE、報酬率誤差、方向準確率、固定猜上漲與前一日價格延續基準。50% 僅為參考線；不以超過 50% 自動宣稱顯著有效或獲利。

In [ ]:
run_section_3_win_rate()

## 4 時光機 歷史與未來

補齊原始缺失函式，顯示 100 日實際歷史及 20 日未來情境。此圖使用稅費前固定股數，正式稅費後績效以歷史回測總表為準。

In [ ]:
run_section_4_time_machine()

## 5 投資組合方向準確率

先將各股真實與預測價格合成相同固定股數組合，再比較每日方向。此數值是方向預測準確率，與完成買賣的交易勝率不同。

In [ ]:
run_section_5_portfolio_win_rate()

## 6 歷史投資組合夏普比率與最大回撤

使用回測期間同一份稅費後淨值計算 Sharpe 及 MDD。

In [ ]:
run_section_6_portfolio_sharpe()

## 7 歷史大盤夏普比率與最大回撤

使用相同期間的 ^TWII 價格指數淨值計算 Sharpe 及 MDD。

In [ ]:
run_section_7_market_sharpe()

## 8 投資組合與大盤日報酬相關係數

依相同日期的每日報酬率計算 Pearson 相關係數；接近零不代表統計獨立。

In [ ]:
run_section_8_portfolio_market_corr()

## 9 研究總表與檔案匯出

依構想書表 3-1 輸出三方策略格式：大盤、舊版、新版。
本版只執行大盤與舊版。新版結果保持空值並標示待串接，不能使用「預期顯著提升」代替實測值。

另輸出原始價格、資料稽核、切分表、每股模型指標、逐日預測、淨值、方向與交易成本、
市場 20 日滾動波動率、訓練歷史，以及重現所需參數。
市場風控門檻使用「截至前一天」的滾動波動率歷史均值加一倍標準差；
歷史不足時維持缺值。此欄位只作為後續串接資料，舊版權重不隨風控訊號更動。

In [ ]:
comparison_table = export_all_results()

## 10 下載執行結果

將 CSV、模型、縮放器及參數檔一起打包。
Colab 會提供 ZIP 下載；本機執行則在目前工作目錄產生 ZIP。
報告與新版比較請使用相同的股票、價格設定、資料版本與測試期間。

In [ ]:
import shutil
archive = shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR)
print("結果壓縮檔：", archive)
try:
    from google.colab import files
except ImportError:
    print("本機執行：請從目前工作目錄開啟上方 ZIP 檔。")
else:
    files.download(archive)

## 交接介面與後續串接

| 已完成的舊版介面 | 回傳資料或用途 |
|---|---|
| `prepare_prices()` | 五檔與大盤共同交易日的收盤價 |
| `train_and_predict_test_set()` | 測試日期、真實收盤價、預測收盤價 |
| `predict_price_sequence()` | 資料截止日與遞迴價格序列 |
| `get_history_and_forecast()` | 歷史與未來日期及價格，供時光機 |
| `get_history_bundle()` | 共用模型、逐日預測、淨值、成本與組合方向 |
| `market_risk_features()` | 波動率與僅使用過去資料的風控門檻 |
| `export_all_results()` | 構想書總表及重現用檔案 |

原始模型預測的是 **下一交易日收盤價**。未來串接的新聞特徵表中
`Target_Next_Return` 是百分比報酬率，不能直接當成此版的價格答案或輸入特徵。
應另建新版多變量模型，並在相同測試日期重新比較。

後續新版才實作新聞情緒、動態權重、單股 30% 配置上限及高波動現金避險。
本次未建立 Streamlit 介面，已提供其可讀取的 CSV 與函式介面。

方法查核資料：
- [scikit-learn 防止資料洩漏](https://scikit-learn.org/stable/common_pitfalls.html)
- [TensorFlow 時序預測與資料切分](https://www.tensorflow.org/tutorials/structured_data/time_series)
- [Keras EarlyStopping](https://keras.io/api/callbacks/early_stopping/)
- [exchange-calendars 支援 XTAI 交易日曆](https://github.com/gerrymanoim/exchange_calendars)
- [證交所臺灣投資指南](https://www.twse.com.tw/zh/about/company/guide.html)
- [證交所營業細則第 94 條 手續費](https://twse-regulation.twse.com.tw/TW/law/DOC01.aspx?FLCODE=FL007304&FLNO=94)